In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys

from torch.utils.data import DataLoader

sys.path.append('PerturbVAE/')

from model import *
from utils import *
from dataset import *
from train import *

In [2]:
data = sc.read_h5ad('data/perturb_filtered_gbm_crispri_crop.h5ad')

#remove treatment NA
na_rows = data.obs[data.obs['treatment'] == 'NA']
data = data[~data.obs.index.isin(na_rows.index)]

In [3]:
dataset = PerturbDataset(data)

In [4]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Example: Iterate through the DataLoader
for X, P, C in dataloader:
    print("Batch X shape:", X.shape)
    print("Batch P:", P)
    print("Batch C:", C)
    break

Batch X shape: torch.Size([32, 2218])
Batch P: tensor([25, 72, 50, 63, 31,  5,  5, 66, 71, 81, 28, 94, 83, 28, 35, 99, 65, 69,
        26, 53, 51, 84, 51, 33, 55,  6, 55, 62, 26, 66, 77, 83])
Batch C: tensor([2, 3, 1, 2, 1, 0, 3, 2, 1, 2, 1, 1, 0, 2, 0, 3, 3, 0, 3, 2, 0, 3, 0, 0,
        3, 2, 2, 0, 0, 0, 1, 1])


In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
from utils import *

In [7]:
data

View of AnnData object with n_obs × n_vars = 18034 × 2218
    obs: 'P7', 'P5', 'sample', 'n.umi', 'cell_ID', 'RT', 'Lig', 'new_cell', 'oligo', 'total_reads.x', 'total_hash_umis_per_cell_ID', 'top_to_second_best_ratio', 'treatment', 'log10.umi', 'percent_mito', 'sgRNA', 'total_reads.y', 'sgRNA_proportion', 'total_sgrna_read_per_cell', 'rank', 'top_to_second', 'second_to_third', 'third_to_next', 'top_proportion', 'second_proportion', 'third_proportion', 'top_sg', 'second_sg', 'third_sg', 'gene_id', 'batch'
    var: 'features'

In [8]:
hyper_config = HyperparamConfig()
hyper_config.n_genes = data.shape[-1]
hyper_config.n_conditions = len(np.unique(dataset.C_indices))
hyper_config.n_perturb = len(np.unique(dataset.P_indices))

In [9]:
hyper_config.n_conditions

4

In [10]:
hyper_config.p_embed_dim

64

In [11]:
config = LitConfig()
model = PerturbVAE(config=hyper_config)
trainer_config = TrainerConfig()
trainer_config.max_epochs = 10
trainer_config.force_cpu = False

In [12]:
data.shape[-1]

2218

In [13]:
train_(model, config, dataloader, val_loader=None, trainer_config=trainer_config)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mingxuan-zhang to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/mingxuanzhang/anaconda3/lib/python3.12/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
/home/mingxuanzhang/anaconda3/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


ValueError: can't optimize a non-leaf Tensor